# Fashion-MNIST Dataset - Model Deployment

### [Fashion-MNIST Dataset](https://github.com/zalandoresearch/fashion-mnist)

Author: [Kevin Thomas](mailto:ket189@pitt.edu)

License: MIT

## Citation

[1] Han Xiao, Kashif Rasul, Roland Vollgraf, https://github.com/zalandoresearch/fashion-mnist

## Install Libraries

In [ ]:
# conda activate prod
# conda install -c conda-forge pytorch torchvision torchaudio
# conda install -c conda-forge onnx onnxruntime
# conda install numpy matplotlib

## Import Libraries

In [ ]:
import argparse
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision import transforms
import onnxruntime
import matplotlib.pyplot as plt
%matplotlib inline

## Seed

In [ ]:
SEED = 42
SEED

In [ ]:
torch.manual_seed(SEED)

## Parameters

In [ ]:
IMAGE_SIZE = 28
IMAGE_SIZE

In [ ]:
NUM_CLASSES = 10
NUM_CLASSES

In [ ]:
CLASS_NAMES = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
CLASS_NAMES

## Hyperparameters

In [ ]:
LEARNING_RATE = 0.001
LEARNING_RATE

In [ ]:
EPOCHS = 3
EPOCHS

In [ ]:
BATCH_SIZE = 128
BATCH_SIZE

## Device

In [ ]:
DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu")
DEVICE

## Load Dataset

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))])
train_data = datasets.FashionMNIST('data', train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST('data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)
len(train_data), len(test_data)

## Create Model

In [ ]:
class CNN(nn.Module):
    """
    A small convolutional neural network for 28x28 grayscale images.
    """

    def __init__(self, num_classes=NUM_CLASSES):
        """
        Initialize the convolutional and dense layers.

        Parameters:
            num_classes (int): Number of output classes.

        Returns:
            None
        """
        super(CNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes))

    def forward(self, x):
        """
        Run the forward pass of the network.

        Parameters:
            x (torch.Tensor): Batch of images.

        Returns:
            torch.Tensor: Output logits.
        """
        x = self.features(x)
        return self.classifier(x)

## Instantiate Model, Loss, and Optimizer

In [ ]:
torch.manual_seed(SEED)
model = CNN().to(DEVICE)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
model

## Train Model

### Functions

In [ ]:
def train_epoch(model, loader, loss_fn, optimizer):
    """
    Train the model for a single epoch.

    Parameters:
        model (nn.Module): The model to train.
        loader (DataLoader): Training data loader.
        loss_fn (nn.Module): Loss function.
        optimizer (torch.optim.Optimizer): Parameter update rule.

    Returns:
        float: Mean training loss.
    """
    model.train()
    total = 0.0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(y)
    return total / len(loader.dataset)


def evaluate(model, loader, loss_fn):
    """
    Evaluate the model over a loader.

    Parameters:
        model (nn.Module): The model to evaluate.
        loader (DataLoader): Evaluation data loader.
        loss_fn (nn.Module): Loss function.

    Returns:
        tuple: Mean loss and accuracy.
    """
    model.eval()
    total, correct = 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            total += loss_fn(logits, y).item() * len(y)
            correct += (logits.argmax(1) == y).sum().item()
    n = len(loader.dataset)
    return total / n, correct / n

### Training Loop

In [ ]:
for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, loss_fn, optimizer)
    test_loss, test_accuracy = evaluate(model, test_loader, loss_fn)
    print(f"Epoch {epoch + 1} | train loss {train_loss:.4f} | test accuracy {test_accuracy:.4f}")

## Save PyTorch Model

In [ ]:
torch.save(model.state_dict(), 'deploy_cnn_fashion_mnist.pt')
print('saved deploy_cnn_fashion_mnist.pt')

## Export to ONNX

ONNX is a portable model format. Exporting makes the model usable outside PyTorch, including in services and on edge devices.

In [ ]:
def export_onnx(model, path):
    """
    Export a PyTorch model to ONNX.

    Parameters:
        model (nn.Module): The trained model.
        path (str): Output ONNX path.

    Returns:
        None
    """
    model.eval()
    dummy = torch.randn(1, 1, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
    torch.onnx.export(
        model, dummy, path,
        input_names=['input'], output_names=['logits'],
        dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
        opset_version=13)

### Run

In [ ]:
export_onnx(model, 'cnn_fashion_mnist.onnx')
print('saved cnn_fashion_mnist.onnx')

## Verify the ONNX Model

The ONNX output must match the PyTorch output. A close match confirms the export is faithful.

In [ ]:
def onnx_logits(path, image):
    """
    Run inference with an ONNX model.

    Parameters:
        path (str): ONNX model path.
        image (numpy.ndarray): A 1x1x28x28 array.

    Returns:
        numpy.ndarray: Output logits.
    """
    session = onnxruntime.InferenceSession(path, providers=['CPUExecutionProvider'])
    return session.run(None, {'input': image.astype(np.float32)})[0]

In [ ]:
image, label = test_data[0]
array = image.unsqueeze(0).numpy()
torch_logits = model(image.unsqueeze(0).to(DEVICE)).detach().cpu().numpy()
onnx_out = onnx_logits('cnn_fashion_mnist.onnx', array)
print('max difference:', np.abs(torch_logits - onnx_out).max())

## Build an Inference Engine

In [ ]:
class InferenceEngine:
    """
    A small inference engine wrapping an ONNX model.
    """

    def __init__(self, path):
        """
        Load the ONNX session.

        Parameters:
            path (str): ONNX model path.

        Returns:
            None
        """
        self.session = onnxruntime.InferenceSession(path, providers=['CPUExecutionProvider'])
        self.class_names = CLASS_NAMES

    def predict(self, image):
        """
        Predict the class for one image tensor.

        Parameters:
            image (torch.Tensor): A 1x28x28 image tensor.

        Returns:
            tuple: Predicted class name and confidence.
        """
        array = image.unsqueeze(0).numpy().astype(np.float32)
        logits = self.session.run(None, {'input': array})[0]
        probs = torch.softmax(torch.tensor(logits), dim=1)
        confidence, predicted = probs.max(dim=1)
        return self.class_names[predicted.item()], confidence.item()

## Inference

In [ ]:
engine = InferenceEngine('cnn_fashion_mnist.onnx')
image, label = test_data[3]
name, confidence = engine.predict(image)
print(f"True: {CLASS_NAMES[label]} | Predicted: {name} ({confidence:.4f})")

## Command-Line Interface

A command-line entry point turns the notebook into a small tool you can run from the terminal.

In [ ]:
def build_parser():
    """
    Build the command-line argument parser.

    Parameters:
        None

    Returns:
        argparse.ArgumentParser: The configured parser.
    """
    parser = argparse.ArgumentParser(description='Classify a Fashion-MNIST image.')
    parser.add_argument('--index', type=int, default=0, help='Test image index.')
    return parser


def main():
    """
    Run the command-line inference engine.

    Parameters:
        None

    Returns:
        None
    """
    args = build_parser().parse_args()
    engine = InferenceEngine('cnn_fashion_mnist.onnx')
    image, label = test_data[args.index]
    name, confidence = engine.predict(image)
    print(f"Index {args.index} | true {CLASS_NAMES[label]} | predicted {name} ({confidence:.4f})")

### Run the CLI

In [ ]:
import sys
sys.argv = ['deploy', '--index', '3']
main()